# Proyecto Final · Bibliotecas públicas de Barcelona
## Notebook 02 — FASE 02: Limpieza y transformación

Partimos de los 6 CSV crudos de `data/raw` y construimos una única tabla maestra
a nivel **distrito-año** (clave compuesta `Codi_Districte` + `Any`).

Plan: limpiar y agregar cada fuente a distrito-año por separado → unirlas todas →
calcular métricas (préstamos per cápita, % mayores, % extranjera) → validar y guardar.

celda 1

In [2]:
#Paso 1 — Montaje - definir rutas RAW/INTERIM/PROCESSED y leer los 6 CSV crudos
#celda 2

import pandas as pd
from pathlib import Path

PROYECTO  = Path.home() / "Desktop" / "proyecto_bibliotecas_bcn"
BASE      = PROYECTO / "data" / "Proyecto Final"   # tu organización por fases
RAW       = BASE / "Fase 01" / "raw"               # donde están los 6 crudos
INTERIM   = BASE / "Fase 02" / "interim"
PROCESSED = BASE / "Fase 02" / "processed"

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print("RAW:", RAW)
print("¿Existen los crudos?", len(list(RAW.glob("*.csv"))), "ficheros")

RAW: C:\Users\User\Desktop\proyecto_bibliotecas_bcn\data\Proyecto Final\Fase 01\raw
¿Existen los crudos? 6 ficheros


### Leer los 6 ficheros crudos

Cargamos cada fuente en su propio DataFrame. Los nombres de variable son claros
para no perdernos: `bib_api`, `bib_csv`, `renta`, `poblacion`, `edad`, `origen`.

celda 3

In [3]:
#celda 4

bib_api   = pd.read_csv(RAW / "bibliotecas_xarxa_2010_2022_raw.csv")
bib_csv   = pd.read_csv(RAW / "bibliotecas_xarxa_2023_2024_raw.csv")
renta     = pd.read_csv(RAW / "renta_disponible_2015_2022_raw.csv")
poblacion = pd.read_csv(RAW / "padron_poblacion_2010_2024_raw.csv")
edad      = pd.read_csv(RAW / "padron_edad_2010_2024_raw.csv")
origen    = pd.read_csv(RAW / "padron_origen_2010_2024_raw.csv")

for nombre, df in [("bib_api", bib_api), ("bib_csv", bib_csv), ("renta", renta),
                   ("poblacion", poblacion), ("edad", edad), ("origen", origen)]:
    print(f"{nombre:11s} {df.shape[0]:>7,} filas  x  {df.shape[1]} columnas")

bib_api       3,720 filas  x  20 columnas
bib_csv         800 filas  x  17 columnas
renta         8,544 filas  x  9 columnas
poblacion    16,020 filas  x  10 columnas
edad        326,762 filas  x  11 columnas
origen       98,827 filas  x  12 columnas


### Paso 2: Bibliotecas → préstamos presenciales por distrito-año

Combinamos los dos bloques (solo columnas comunes que necesitamos), filtramos el
indicador `Prestecs_presencials` y agregamos sumando los préstamos de todas las
bibliotecas de cada distrito y año.

celda 5

In [4]:
#Paso 2 — Bibliotecas
# celda 6

# Solo las columnas comunes que nos interesan (evita el lío de columnas sobrantes)
cols = ["Any", "Codi_Districte", "Nom_Districte", "Nom_Equipament", "Indicador", "Valor"]
bib = pd.concat([bib_api[cols], bib_csv[cols]], ignore_index=True)

# Nos quedamos solo con el préstamo presencial
bib_pres = bib[bib["Indicador"] == "Prestecs_presencials"].copy()

print("Filas tras filtrar:", len(bib_pres))
print("Años:", sorted(bib_pres["Any"].unique()))
print("Códigos de distrito:", sorted(bib_pres["Codi_Districte"].unique()))
print("Tipo de 'Valor':", bib_pres["Valor"].dtype, "| Nulos:", bib_pres["Valor"].isna().sum())

Filas tras filtrar: 590
Años: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Códigos de distrito: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(nan)]
Tipo de 'Valor': int64 | Nulos: 0


In [5]:
#celda 7 nueva

# Código (diagnóstico: ¿qué filas no tienen distrito?)

sin_distrito = bib_pres[bib_pres["Codi_Districte"].isna()]

print("Filas sin distrito:", len(sin_distrito))
print("Bibliotecas afectadas:", sin_distrito["Nom_Equipament"].unique())
print("Años:", sorted(sin_distrito["Any"].unique()))
print("Préstamos sin distrito:", sin_distrito["Valor"].sum())
print("% sobre el total:", round(100 * sin_distrito["Valor"].sum() / bib_pres["Valor"].sum(), 2), "%")

Filas sin distrito: 40
Bibliotecas afectadas: [nan]
Años: [np.int64(2020)]
Préstamos sin distrito: 1702158
% sobre el total: 3.31 %


In [6]:
#celda 7b !Problema de los registros sin distrito.

resumen = bib_pres.groupby("Any").agg(
    filas=("Valor", "size"),
    con_nombre=("Nom_Equipament", lambda s: s.notna().sum()),
)
print(resumen)

      filas  con_nombre
Any                    
2010     35          35
2011     37          37
2012     39          39
2013     39          39
2014     40          40
2015     40          40
2016     40          40
2017     40          40
2018     40          40
2019     40          40
2020     40           0
2021     40          40
2022     40          40
2023     40          40
2024     40          40


In [7]:
#celda 7c  !Problema de los registros sin distrito.

#Código (investigar las filas de 2020 en el crudo completo)

# Filas de 2020 (préstamo presencial) con TODAS las columnas originales
mask2020 = (bib_api["Any"] == 2020) & (bib_api["Indicador"] == "Prestecs_presencials")
sub2020 = bib_api.loc[mask2020]

print("Filas 2020:", len(sub2020))
print("\nNo-nulos por columna (qué información SÍ tienen esas filas):")
print(sub2020.notna().sum())

Filas 2020: 40

No-nulos por columna (qué información SÍ tienen esas filas):
Latitud             40
Titularitat         40
Codi_Districte       0
Tipus_Us             0
Notes_Equipament     1
Nom_Districte       40
Notes_Dades         40
Nom_Barri           40
Indicador           40
Tipus_Equipament     0
Nom_Equipament       0
Valor               40
Codi_Barri           0
Ambit               40
_id                 40
Any                 40
Longitud            40
any_recurso         40
TipusGeneral        40
Equipament          40
dtype: int64


In [8]:
#celda 7d  !Problema de los registros sin distrito.

# Código (¿coinciden los nombres de distrito?)

# Nombres de distrito en las filas que SÍ tienen código (años buenos)
nombres_ok = set(bib_pres.dropna(subset=["Codi_Districte"])["Nom_Districte"].unique())

# Nombres de distrito en las filas SIN código (las de 2020 a recuperar)
nombres_2020 = set(bib_pres[bib_pres["Codi_Districte"].isna()]["Nom_Districte"].unique())

print("Distritos en años con código:", len(nombres_ok))
print(sorted(nombres_ok))
print("\nDistritos en las filas de 2020:", len(nombres_2020))
print(sorted(nombres_2020))

print("\nNombres en 2020 que NO aparecen en los años buenos:")
print(nombres_2020 - nombres_ok)

Distritos en años con código: 10
['Ciutat Vella', 'Eixample', 'Gràcia', 'Horta-Guinardó', 'Les Corts', 'Nou Barris', 'Sant Andreu', 'Sant Martí', 'Sants-Montjuïc', 'Sarrià-Sant Gervasi']

Distritos en las filas de 2020: 10
['01. Ciutat Vella', '02. Eixample', '03. Sants-Montjuïc', '04. Les Corts', '05. Sarrià-Sant Gervasi', '06. Gràcia', '07. Horta-Guinardó', '08. Nou Barris', '09. Sant Andreu', '10. Sant Martí']

Nombres en 2020 que NO aparecen en los años buenos:
{'07. Horta-Guinardó', '08. Nou Barris', '02. Eixample', '03. Sants-Montjuïc', '06. Gràcia', '01. Ciutat Vella', '09. Sant Andreu', '10. Sant Martí', '05. Sarrià-Sant Gervasi', '04. Les Corts'}


In [9]:
# #celda 7e  !Problema de los registros sin distrito. - SOLUCIONADO

# Código (rescatar 2020 desde el prefijo del nombre)
# Sacar el código del prefijo "0X." del nombre (solo hay prefijo en las filas de 2020)
codigo_prefijo = bib_pres["Nom_Districte"].str.extract(r"^\s*(\d+)", expand=False)

# Rellenar SOLO los códigos que faltan (2020) con ese número
bib_pres["Codi_Districte"] = bib_pres["Codi_Districte"].fillna(
    pd.to_numeric(codigo_prefijo, errors="coerce"))

# Normalizar el nombre: quitar el prefijo "0X. " para que 2020 case con los demás años
bib_pres["Nom_Districte"] = bib_pres["Nom_Districte"].str.replace(r"^\s*\d+\.\s*", "", regex=True)

# Verificar que el problema está resuelto
print("Nulos restantes en Codi_Districte:", bib_pres["Codi_Districte"].isna().sum())
print("Nombres únicos:", sorted(bib_pres["Nom_Districte"].unique()))

Nulos restantes en Codi_Districte: 0
Nombres únicos: ['Ciutat Vella', 'Eixample', 'Gràcia', 'Horta-Guinardó', 'Les Corts', 'Nou Barris', 'Sant Andreu', 'Sant Martí', 'Sants-Montjuïc', 'Sarrià-Sant Gervasi']


In [10]:
#celda 8

# Red de seguridad: si quedara algún código sin recuperar, se descarta. Pasamos a entero.
bib_pres = bib_pres.dropna(subset=["Codi_Districte"]).copy()
bib_pres["Codi_Districte"] = bib_pres["Codi_Districte"].astype(int)

print("Filas tras limpiar:", len(bib_pres))
print("Distritos:", sorted(bib_pres["Codi_Districte"].unique()))

Filas tras limpiar: 590
Distritos: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


In [11]:
#celda 9
bib_pres["Valor"] = pd.to_numeric(bib_pres["Valor"], errors="coerce")

prestamos = (bib_pres
    .groupby(["Codi_Districte", "Nom_Districte", "Any"], as_index=False)["Valor"]
    .sum()
    .rename(columns={"Valor": "prestamos_presenciales"}))

print("Tabla préstamos por distrito-año:", prestamos.shape)
prestamos.head(12)

Tabla préstamos por distrito-año: (150, 4)


,Codi_Districte,Nom_Districte,Any,prestamos_presenciales
0,1,Ciutat Vella,2010,109276
1,1,Ciutat Vella,2011,449563
2,1,Ciutat Vella,2012,427758
3,1,Ciutat Vella,2013,346104
4,1,Ciutat Vella,2014,391455
5,1,Ciutat Vella,2015,370390
6,1,Ciutat Vella,2016,326033
7,1,Ciutat Vella,2017,295995
8,1,Ciutat Vella,2018,273041
9,1,Ciutat Vella,2019,268114


### Guardar bibliotecas agregado (interim)

`prestamos` ya está a nivel distrito-año. La ordenamos (solo por estética) y la
guardamos en `data/interim`. Es una tabla intermedia: aún falta unirla con
población, renta y demografía.

In [12]:
# Orden solo para legibilidad (no afecta a los cálculos)
prestamos = prestamos.sort_values(["Codi_Districte", "Any"]).reset_index(drop=True)

ruta = INTERIM / "prestamos_distrito_anyo.csv"
prestamos.to_csv(ruta, index=False, encoding="utf-8")

print("Guardado:", ruta)
print("Filas:", len(prestamos), "| Distritos:", prestamos["Codi_Districte"].nunique(),
      "| Años:", prestamos["Any"].nunique())

Guardado: C:\Users\User\Desktop\proyecto_bibliotecas_bcn\data\Proyecto Final\Fase 02\interim\prestamos_distrito_anyo.csv
Filas: 150 | Distritos: 10 | Años: 15


## Paso 3: Población → total por distrito-año

Del padrón a nivel sección censal, sumamos por distrito y año para obtener la
población total de cada distrito (denominador de "préstamos per cápita").
Primero inspeccionamos códigos y nulos; luego agregamos y validamos el total.

celda 12

In [13]:
#Paso 3 — Población
#celda 13

print("Columnas:", list(poblacion.columns))
print("Códigos de distrito:", sorted(poblacion["Codi_Districte"].dropna().unique()))
print("Nulos en Codi_Districte:", poblacion["Codi_Districte"].isna().sum())
print("Años (any_recurso):", sorted(poblacion["any_recurso"].unique()))
print("Tipo de Valor:", poblacion["Valor"].dtype, "| Nulos:", poblacion["Valor"].isna().sum())

Columnas: ['Codi_Districte', 'Nom_Districte', 'Codi_Barri', 'Nom_Barri', 'AEB', 'Seccio_Censal', 'Valor', 'Data_Referencia', '_id', 'any_recurso']
Códigos de distrito: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Nulos en Codi_Districte: 0
Años (any_recurso): [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Tipo de Valor: int64 | Nulos: 0


In [14]:
#celda 14

poblacion["Valor"] = pd.to_numeric(poblacion["Valor"], errors="coerce")

pob_distrito = (poblacion
    .groupby(["Codi_Districte", "Nom_Districte", "any_recurso"], as_index=False)["Valor"]
    .sum()
    .rename(columns={"any_recurso": "Any", "Valor": "poblacion"}))

print("Tabla población distrito-año:", pob_distrito.shape)
print("\nPoblación total de Barcelona por año (control de coherencia):")
print(pob_distrito.groupby("Any")["poblacion"].sum())

Tabla población distrito-año: (150, 4)

Población total de Barcelona por año (control de coherencia):
Any
2010    1619613
2011    1614589
2012    1619830
2013    1612835
2014    1602450
2015    1604700
2016    1610427
2017    1625137
2018    1628936
2019    1650358
2020    1666530
2021    1660314
2022    1639981
2023    1660435
2024    1702814
Name: poblacion, dtype: int64


### Guardar población agregada (interim)

`pob_distrito` ya está a nivel distrito-año (población total por distrito, el
denominador de "préstamos per cápita"). La guardamos en `data/interim`. El detalle
por sección/barrio sigue en el raw para calcular los pesos de la renta en el Paso 4.

celda 15

In [15]:
#celda 16

pob_distrito = pob_distrito.sort_values(["Codi_Districte", "Any"]).reset_index(drop=True)

ruta = INTERIM / "poblacion_distrito_anyo.csv"
pob_distrito.to_csv(ruta, index=False, encoding="utf-8")

print("Guardado:", ruta)
print("Filas:", len(pob_distrito), "| Años:", pob_distrito["Any"].nunique())

Guardado: C:\Users\User\Desktop\proyecto_bibliotecas_bcn\data\Proyecto Final\Fase 02\interim\poblacion_distrito_anyo.csv
Filas: 150 | Años: 15


## Paso 4: Renta → media ponderada por distrito-año

La renta viene por sección censal (per cápita). Como las secciones no casan con
el padrón, agregamos al nivel común (barrio) y calculamos la renta de distrito
como media de barrios ponderada por su población. Cobertura: 2015–2022.

#celda 17

In [16]:
#celda 18

print("Columnas renta:", list(renta.columns))
print("Distritos:", sorted(renta["Codi_Districte"].dropna().unique()))
print("Nulos en Codi_Districte:", renta["Codi_Districte"].isna().sum())
print("Años:", sorted(renta["any_recurso"].unique()))
print("Import_Euros -> tipo:", renta["Import_Euros"].dtype, "| Nulos:", renta["Import_Euros"].isna().sum())

# ¿Los barrios de renta existen en población? (clave para que el puente funcione)
b_renta = set(renta[["Codi_Districte", "Codi_Barri"]].dropna()
                   .drop_duplicates().itertuples(index=False, name=None))
b_pob   = set(poblacion[["Codi_Districte", "Codi_Barri"]].dropna()
                   .drop_duplicates().itertuples(index=False, name=None))
print("\nBarrios en renta:", len(b_renta), "| en población:", len(b_pob))
print("Barrios de renta que NO están en población:", len(b_renta - b_pob))

Columnas renta: ['Codi_Districte', 'Import_Euros', 'Nom_Districte', 'Nom_Barri', 'Seccio_Censal', 'Codi_Barri', '_id', 'Any', 'any_recurso']
Distritos: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Nulos en Codi_Districte: 0
Años: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Import_Euros -> tipo: int64 | Nulos: 0

Barrios en renta: 73 | en población: 73
Barrios de renta que NO están en población: 0


In [17]:
#celda 19

# A) Peso: población por barrio-año (suma de sus secciones)
pob_barrio = (poblacion
    .groupby(["Codi_Districte", "Codi_Barri", "any_recurso"], as_index=False)["Valor"]
    .sum()
    .rename(columns={"Valor": "pob_barrio"}))

# B) Renta por barrio-año (media de las secciones del barrio)
renta_barrio = (renta
    .groupby(["Codi_Districte", "Codi_Barri", "any_recurso"], as_index=False)["Import_Euros"]
    .mean()
    .rename(columns={"Import_Euros": "renta_barrio"}))

# C) Unir renta + peso por barrio-año (inner -> solo años comunes, 2015-2022)
barrio = renta_barrio.merge(pob_barrio, on=["Codi_Districte", "Codi_Barri", "any_recurso"], how="inner")

print("Barrios-año combinados:", barrio.shape)   # ~73 barrios x 8 años = 584
print("Años:", sorted(barrio["any_recurso"].unique()))
barrio.head()

Barrios-año combinados: (584, 5)
Años: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]


,Codi_Districte,Codi_Barri,any_recurso,renta_barrio,pob_barrio
0,1,1,2015,11834.904762,47150
1,1,1,2016,12020.142857,47196
2,1,1,2017,12559.333333,47901
3,1,1,2018,12752.857143,47514
4,1,1,2019,13125.047619,48210


In [18]:
#celda 20

# Numerador de la media ponderada: renta de cada barrio * su población
barrio["renta_x_pob"] = barrio["renta_barrio"] * barrio["pob_barrio"]

# Por distrito-año: sumamos (renta*pob) y sumamos pob, y dividimos
renta_distrito = (barrio
    .groupby(["Codi_Districte", "any_recurso"], as_index=False)
    .agg(suma_rxp=("renta_x_pob", "sum"), suma_pob=("pob_barrio", "sum")))

renta_distrito["renta_per_capita"] = (renta_distrito["suma_rxp"] / renta_distrito["suma_pob"]).round(0)

renta_distrito = (renta_distrito
    .rename(columns={"any_recurso": "Any"})[["Codi_Districte", "Any", "renta_per_capita"]])

print("Renta ponderada por distrito-año:", renta_distrito.shape)   # esperado 10 x 8 = 80
renta_distrito.head(12)

Renta ponderada por distrito-año: (80, 3)


,Codi_Districte,Any,renta_per_capita
0,1,2015,13831.0
1,1,2016,14193.0
2,1,2017,14751.0
3,1,2018,15036.0
4,1,2019,15665.0
5,1,2020,14019.0
6,1,2021,15382.0
7,1,2022,16831.0
8,2,2015,23025.0
9,2,2016,23900.0


In [19]:
#celda 21 - Validacion rapida de la renta en 2022

print("Renta per cápita por distrito en 2022 (de mayor a menor):")
print(renta_distrito[renta_distrito["Any"] == 2022]
      .sort_values("renta_per_capita", ascending=False)
      .to_string(index=False))

Renta per cápita por distrito en 2022 (de mayor a menor):
 Codi_Districte  Any  renta_per_capita
              5 2022           35076.0
              4 2022           29497.0
              2 2022           25994.0
              6 2022           25269.0
             10 2022           21408.0
              7 2022           20677.0
              9 2022           20514.0
              3 2022           20357.0
              8 2022           16935.0
              1 2022           16831.0


### Guardar renta agregada (interim)

`renta_distrito` es la renta per cápita por distrito-año, calculada como media de
barrios ponderada por población (2015–2022). La guardamos en `data/interim`.

celda 22

In [20]:
#celda 23

renta_distrito = renta_distrito.sort_values(["Codi_Districte", "Any"]).reset_index(drop=True)

ruta = INTERIM / "renta_distrito_anyo.csv"
renta_distrito.to_csv(ruta, index=False, encoding="utf-8")

print("Guardado:", ruta)
print("Filas:", len(renta_distrito), "| Años:", sorted(renta_distrito["Any"].unique()))

Guardado: C:\Users\User\Desktop\proyecto_bibliotecas_bcn\data\Proyecto Final\Fase 02\interim\renta_distrito_anyo.csv
Filas: 80 | Años: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]


In [21]:
#celda 23b Este ejemplo me ha ayudado a consolidar la idea de la media simple vs la media ponderada. 

#  Ejemplo real: los barrios de Ciutat Vella (distrito 1) en 2015
ej = barrio[(barrio["Codi_Districte"] == 1) & (barrio["any_recurso"] == 2015)].copy()

print("Barrios de Ciutat Vella en 2015:")
print(ej[["Codi_Barri", "renta_barrio", "pob_barrio"]].to_string(index=False))

media_simple    = ej["renta_barrio"].mean()
num             = (ej["renta_barrio"] * ej["pob_barrio"]).sum()
den             = ej["pob_barrio"].sum()
media_ponderada = num / den

print(f"\nMedia SIMPLE (todos los barrios igual): {media_simple:,.0f} €")
print(f"Σ(renta×pob) = {num:,.0f}")
print(f"Σ(pob)       = {den:,.0f}")
print(f"Media PONDERADA = Σ(renta×pob)/Σ(pob) = {media_ponderada:,.0f} €")

#

Barrios de Ciutat Vella en 2015:
 Codi_Barri  renta_barrio  pob_barrio
          1  11834.904762       47150
          2  15615.666667       15514
          3  14988.181818       15037
          4  16014.153846       22468

Media SIMPLE (todos los barrios igual): 14,613 €
Σ(renta×pob) = 1,385,460,511
Σ(pob)       = 100,169
Media PONDERADA = Σ(renta×pob)/Σ(pob) = 13,831 €


## Paso 5: Edad → % de población mayor (65+) por distrito-año

Del padrón por grupos quinquenales. Primero identificamos la columna de grupo de
edad y sus etiquetas para saber cuáles son "65 y más". Luego, por distrito-año:
% mayores = población de 65+ / población total (ambas del mismo dataset).

celda 24

In [22]:
#celda 25

print("Columnas y nº de valores únicos:")
for col in edad.columns:
    print(f"  {col}: {edad[col].nunique()}")

print("\n--- Valores de columnas con pocos únicos (buscando los grupos de edad) ---")
for col in edad.columns:
    nu = edad[col].nunique()
    if 5 <= nu <= 25:
        print(f"\n{col} ({nu} únicos):")
        print(list(edad[col].dropna().unique())[:30])

Columnas y nº de valores únicos:
  Codi_Districte: 10
  Nom_Districte: 10
  Codi_Barri: 73
  Nom_Barri: 73
  AEB: 233
  Seccio_Censal: 1068
  Valor: 560


  Data_Referencia: 15
  _id: 21951
  EDAT_Q: 22
  any_recurso: 15

--- Valores de columnas con pocos únicos (buscando los grupos de edad) ---

Codi_Districte (10 únicos):
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]

Nom_Districte (10 únicos):
['Ciutat Vella', 'Eixample', 'Sants-Montjuïc', 'Les Corts', 'Sarrià-Sant Gervasi', 'Gràcia', 'Horta-Guinardó', 'Nou Barris', 'Sant Andreu', 'Sant Martí']

Data_Referencia (15 únicos):
['2010-01-01T00:00:00', '2011-01-01T00:00:00', '2012-01-01T00:00:00', '2013-01-01T00:00:00', '2014-01-01T00:00:00', '2015-01-01T00:00:00', '2016-01-01T00:00:00', '2017-01-01T00:00:00', '2018-01-01T00:00:00', '2019-01-01T00:00:00', '2020-01-01T00:00:00', '2021-01-01T00:00:00', '2022-01-01T00:00:00', '2023-01-01T00:00:00', '2024-01-01T00:00:00']

EDAT_Q (22 únicos):
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), 

### Identificar 65+ y validar

EDAT_Q es un código quinquenal (0=0-4, 1=5-9, ... k=[5k, 5k+4]). Por tanto
"65 y más" = códigos >= 13. Validamos el mapeo contra el dato real: en Barcelona
el ~21-22% de la población tiene 65+.

celda 26

In [23]:
#celda 27

print("dtype original de Valor:", edad["Valor"].dtype)

# ¿Cuántos valores no son numéricos y cómo son?
no_num = pd.to_numeric(edad["Valor"], errors="coerce").isna()
print("Valores no numéricos:", no_num.sum(), f"({100*no_num.mean():.2f}% del total)")
print("Ejemplos:", edad.loc[no_num, "Valor"].unique()[:10])

# Convertimos a numérico (los suprimidos -> NaN; en las sumas se ignoran)
edad["Valor"] = pd.to_numeric(edad["Valor"], errors="coerce")

print("\nPoblación por código de edad (ya en números):")
print(edad.groupby("EDAT_Q")["Valor"].sum())

dtype original de Valor: object
Valores no numéricos: 16354 (5.00% del total)
Ejemplos: ['..']

Población por código de edad (ya en números):
EDAT_Q
0      996026.0
1     1011913.0
2      996290.0
3     1012436.0
4     1234463.0
5     1730153.0
6     1989959.0
7     2009949.0
8     1931482.0
9     1811658.0
10    1670672.0
11    1528941.0
12    1402015.0
13    1274473.0
14    1124258.0
15    1005112.0
16     838706.0
17     595337.0
18     271053.0
19      48950.0
20        523.0
21          0.0
Name: Valor, dtype: float64


### Calcular % mayores (65+) y % jóvenes (0-29) por distrito-año

En vez de una edad media/mediana (que sobre datos agrupados son aproximaciones),
usamos porcentajes exactos: recuentos de cada tramo / población total. Dos extremos
de la pirámide para la PI3:
- % mayores = población 65+ (EDAT_Q >= 13) / total
- % jóvenes = población 0-29 (EDAT_Q <= 5) / total

Validamos el mapeo de edad contra el dato real: en Barcelona el ~21-22% tiene 65+.

celda 28

In [24]:
#celda 29

edad["Valor"] = pd.to_numeric(edad["Valor"], errors="coerce")

tot = (edad.groupby(["Codi_Districte", "any_recurso"], as_index=False)["Valor"]
          .sum().rename(columns={"Valor": "pob_total"}))

mayores = (edad[edad["EDAT_Q"] >= 13]                 # 65 y más (códigos 13-21)
           .groupby(["Codi_Districte", "any_recurso"], as_index=False)["Valor"]
           .sum().rename(columns={"Valor": "pob_65mas"}))

jovenes = (edad[edad["EDAT_Q"] <= 5]                  # 0-29 años (códigos 0-5)
           .groupby(["Codi_Districte", "any_recurso"], as_index=False)["Valor"]
           .sum().rename(columns={"Valor": "pob_joven"}))

edad_distrito = (tot
    .merge(mayores, on=["Codi_Districte", "any_recurso"])
    .merge(jovenes, on=["Codi_Districte", "any_recurso"]))

edad_distrito["pct_mayores"] = (100 * edad_distrito["pob_65mas"] / edad_distrito["pob_total"]).round(1)
edad_distrito["pct_joven"]   = (100 * edad_distrito["pob_joven"] / edad_distrito["pob_total"]).round(1)
edad_distrito = edad_distrito.rename(columns={"any_recurso": "Any"})

print("Forma:", edad_distrito.shape, "(esperado 150)")

# Validación del % 65+ por año (~21-22% esperado)
bcn   = edad.groupby("any_recurso")["Valor"].sum()
bcn65 = edad[edad["EDAT_Q"] >= 13].groupby("any_recurso")["Valor"].sum()
print("% 65+ Barcelona por año:", (100 * bcn65 / bcn).round(1).tolist())

edad_distrito.head()

Forma: (150, 7) (esperado 150)
% 65+ Barcelona por año: [20.4, 20.7, 20.8, 21.0, 21.3, 21.5, 21.5, 21.4, 21.3, 21.1, 21.0, 21.0, 21.2, 21.0, 20.8]


,Codi_Districte,Any,pob_total,pob_65mas,pob_joven,pct_mayores,pct_joven
0,1,2010,104544.0,16305.0,32659.0,15.6,31.2
1,1,2011,103455.0,15886.0,32341.0,15.4,31.3
2,1,2012,104260.0,15605.0,32471.0,15.0,31.1
3,1,2013,103193.0,15197.0,31723.0,14.7,30.7
4,1,2014,100571.0,14735.0,30622.0,14.7,30.4


### Guardar edad agregada (interim)

`edad_distrito` está a nivel distrito-año con % mayores (65+) y % jóvenes (0-29),
validados contra el dato real de Barcelona. La guardamos en `data/interim`.

celda 30

In [25]:
#celda 31

edad_distrito = edad_distrito.sort_values(["Codi_Districte", "Any"]).reset_index(drop=True)

ruta = INTERIM / "edad_distrito_anyo.csv"
edad_distrito.to_csv(ruta, index=False, encoding="utf-8")

print("Guardado:", ruta)
print("Filas:", len(edad_distrito), "| Columnas:", list(edad_distrito.columns))

Guardado: C:\Users\User\Desktop\proyecto_bibliotecas_bcn\data\Proyecto Final\Fase 02\interim\edad_distrito_anyo.csv
Filas: 150 | Columnas: ['Codi_Districte', 'Any', 'pob_total', 'pob_65mas', 'pob_joven', 'pct_mayores', 'pct_joven']


## Paso 6: Origen → % de población extranjera por distrito-año

Del padrón por nacionalidad (Espanya / UE / Resta). Identificamos las categorías
de NACIONALITAT_G, sumamos sobre SEXE, y calculamos:
% extranjera = población no española / población total, por distrito-año.

celda 32

In [26]:
#celda 33

print("Columnas:", list(origen.columns))
print("\nNACIONALITAT_G (categorías):", origen["NACIONALITAT_G"].unique())
print("SEXE (categorías):", origen["SEXE"].unique())
print("Distritos:", sorted(origen["Codi_Districte"].dropna().unique()))
print("Años:", sorted(origen["any_recurso"].unique()))

print("\nValor -> dtype:", origen["Valor"].dtype)
no_num = pd.to_numeric(origen["Valor"], errors="coerce").isna()
print("Valores no numéricos:", no_num.sum(), f"({100*no_num.mean():.2f}%)",
      "| ejemplos:", list(origen.loc[no_num, "Valor"].unique())[:5])

Columnas: ['Codi_Districte', 'SEXE', 'NACIONALITAT_G', 'Nom_Districte', 'Codi_Barri', 'Nom_Barri', 'AEB', 'Seccio_Censal', 'Valor', 'Data_Referencia', '_id', 'any_recurso']

NACIONALITAT_G (categorías): [1 2 3 4]
SEXE (categorías): [1 2]
Distritos: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Años: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Valor -> dtype: object
Valores no numéricos: 3292 (3.33%) | ejemplos: ['..']


### Identificar qué código de nacionalidad es "española"

NACIONALITAT_G tiene 4 códigos [1, 2, 3, 4] y no sabemos su significado. La
nacionalidad española es la mayoritaria (~75-80% en Barcelona), así que el código
con más población será España. El resto (sumados) = población extranjera (~20-25%),
que de paso valida el reparto contra el dato real de la ciudad.

celda 34

In [27]:
#celda 35

origen["Valor"] = pd.to_numeric(origen["Valor"], errors="coerce")

reparto = origen.groupby("NACIONALITAT_G")["Valor"].sum()
print("Población total por código de nacionalidad (toda la serie):")
print(reparto)
print("\n% sobre el total:")
print((100 * reparto / reparto.sum()).round(1))

Población total por código de nacionalidad (toda la serie):
NACIONALITAT_G
1    19744828.0
2     1358012.0
3     3411049.0
4          41.0
Name: Valor, dtype: float64

% sobre el total:
NACIONALITAT_G
1    80.5
2     5.5
3    13.9
4     0.0
Name: Valor, dtype: float64


### Calcular % de población extranjera por distrito-año

Confirmado que el código 1 = España (80,5%). "Extranjera" = NACIONALITAT_G != 1.
Por distrito-año: % extranjera = población no española / población total
(sumando ambos sexos). Validamos contra el ~17-22% real de Barcelona.

celda 36

In [28]:
#celda 37

# Población total por distrito-año (suma todas las nacionalidades y sexos)
tot = (origen.groupby(["Codi_Districte", "any_recurso"], as_index=False)["Valor"]
          .sum().rename(columns={"Valor": "pob_total"}))

# Población extranjera (todo lo que NO es código 1 = España)
ext = (origen[origen["NACIONALITAT_G"] != 1]
       .groupby(["Codi_Districte", "any_recurso"], as_index=False)["Valor"]
       .sum().rename(columns={"Valor": "pob_extranjera"}))

origen_distrito = tot.merge(ext, on=["Codi_Districte", "any_recurso"])
origen_distrito["pct_extranjera"] = (100 * origen_distrito["pob_extranjera"] / origen_distrito["pob_total"]).round(1)
origen_distrito = origen_distrito.rename(columns={"any_recurso": "Any"})

print("Forma:", origen_distrito.shape, "(esperado 150)")

# Validación: % extranjera de Barcelona por año (~17-22%)
bcn    = origen.groupby("any_recurso")["Valor"].sum()
bcnext = origen[origen["NACIONALITAT_G"] != 1].groupby("any_recurso")["Valor"].sum()
print("% extranjera Barcelona por año:", (100 * bcnext / bcn).round(1).tolist())

origen_distrito.head()

Forma: (150, 5) (esperado 150)
% extranjera Barcelona por año: [17.5, 17.3, 17.5, 17.4, 17.0, 16.4, 16.6, 17.7, 18.5, 20.2, 21.6, 22.4, 22.2, 23.6, 25.4]


,Codi_Districte,Any,pob_total,pob_extranjera,pct_extranjera
0,1,2010,104643.0,43216.0,41.3
1,1,2011,103549.0,42696.0,41.2
2,1,2012,104334.0,44018.0,42.2
3,1,2013,103285.0,44491.0,43.1
4,1,2014,100678.0,43300.0,43.0


### Guardar origen agregado (interim)

`origen_distrito` a nivel distrito-año con % de población extranjera, validado
contra la evolución real de Barcelona (baja en la crisis, sube hasta ~25% en 2024).
La guardamos en `data/interim`. Con esto, todas las fuentes quedan procesadas.

celda 38

In [29]:
#celda 39

origen_distrito = origen_distrito.sort_values(["Codi_Districte", "Any"]).reset_index(drop=True)

ruta = INTERIM / "origen_distrito_anyo.csv"
origen_distrito.to_csv(ruta, index=False, encoding="utf-8")

print("Guardado:", ruta)
print("Filas:", len(origen_distrito), "| Columnas:", list(origen_distrito.columns))

Guardado: C:\Users\User\Desktop\proyecto_bibliotecas_bcn\data\Proyecto Final\Fase 02\interim\origen_distrito_anyo.csv
Filas: 150 | Columnas: ['Codi_Districte', 'Any', 'pob_total', 'pob_extranjera', 'pct_extranjera']


## Paso 7: Unión final → tabla maestra distrito-año

Unimos las 5 intermedias por (Codi_Districte, Any) con left join sobre el esqueleto
de préstamos (2010-2024). La renta solo existe 2015-2022, así que fuera de esa
ventana quedará como NaN (esperado). Luego calculamos préstamos per cápita e
índice base 100.

celda 40

In [30]:
#celda 41 

# Esqueleto: préstamos (150 filas, 2010-2024) con distrito y nombre
maestra = prestamos[["Codi_Districte", "Nom_Districte", "Any", "prestamos_presenciales"]].copy()

# Unimos cada fuente con solo sus columnas útiles (left join para conservar 2010-2024)
maestra = maestra.merge(pob_distrito[["Codi_Districte", "Any", "poblacion"]],
                        on=["Codi_Districte", "Any"], how="left")
maestra = maestra.merge(edad_distrito[["Codi_Districte", "Any", "pct_mayores", "pct_joven"]],
                        on=["Codi_Districte", "Any"], how="left")
maestra = maestra.merge(origen_distrito[["Codi_Districte", "Any", "pct_extranjera"]],
                        on=["Codi_Districte", "Any"], how="left")
maestra = maestra.merge(renta_distrito[["Codi_Districte", "Any", "renta_per_capita"]],
                        on=["Codi_Districte", "Any"], how="left")

print("Tabla maestra:", maestra.shape)
print("\nNulos por columna:")
print(maestra.isna().sum())
maestra.head()

Tabla maestra: (150, 9)

Nulos por columna:
Codi_Districte             0
Nom_Districte              0
Any                        0
prestamos_presenciales     0
poblacion                  0
pct_mayores                0
pct_joven                  0
pct_extranjera             0
renta_per_capita          70
dtype: int64


,Codi_Districte,Nom_Districte,Any,prestamos_presenciales,poblacion,pct_mayores,pct_joven,pct_extranjera,renta_per_capita
0,1,Ciutat Vella,2010,109276,104664,15.6,31.2,41.3,NaN
1,1,Ciutat Vella,2011,449563,103569,15.4,31.3,41.2,NaN
2,1,Ciutat Vella,2012,427758,104357,15.0,31.1,42.2,NaN
3,1,Ciutat Vella,2013,346104,103311,14.7,30.7,43.1,NaN
4,1,Ciutat Vella,2014,391455,100700,14.7,30.4,43.0,NaN


In [33]:
#celda 42

# Préstamos per cápita (préstamos por habitante)
maestra["prestamos_per_capita"] = (maestra["prestamos_presenciales"] / maestra["poblacion"]).round(2)

# Índice base 100, con base 2011 (2010 está infrarreportado)
ANYO_BASE = 2011
base = maestra[maestra["Any"] == ANYO_BASE].set_index("Codi_Districte")["prestamos_presenciales"]
maestra["indice_base100"] = (100 * maestra["prestamos_presenciales"] / maestra["Codi_Districte"].map(base)).round(1)

print("Tabla maestra final:", maestra.shape)
print("Índice en", ANYO_BASE, "(debe ser 100.0):", maestra.loc[maestra["Any"] == ANYO_BASE, "indice_base100"].unique())
maestra[maestra["Codi_Districte"] == 1][["Any", "prestamos_presenciales", "prestamos_per_capita", "indice_base100"]]

Tabla maestra final: (150, 11)
Índice en 2011 (debe ser 100.0): [100.]


,Any,prestamos_presenciales,prestamos_per_capita,indice_base100
0,2010,109276,1.04,24.3
1,2011,449563,4.34,100.0
2,2012,427758,4.10,95.1
3,2013,346104,3.35,77.0
4,2014,391455,3.89,87.1
5,2015,370390,3.70,82.4
6,2016,326033,3.25,72.5
7,2017,295995,2.90,65.8
8,2018,273041,2.68,60.7
9,2019,268114,2.54,59.6


In [34]:
#celda 43

#  1) El índice base 100 en 2010 debe ser 100.0 en todos los distritos
print("Índice base 100 en 2010 (debe ser solo 100.0):")
print(maestra.loc[maestra["Any"] == 2010, "indice_base100"].unique())

# 2) ¿Es 2010 anómalamente bajo? Total de préstamos de Barcelona por año
print("\nPréstamos totales de Barcelona por año:")
print(maestra.groupby("Any")["prestamos_presenciales"].sum())

Índice base 100 en 2010 (debe ser solo 100.0):
[24.3 30.  37.4 37.6 32.4 32.  35.7 30.4 32.6 35. ]

Préstamos totales de Barcelona por año:
Any
2010    1529116
2011    4760341
2012    4451296
2013    3669154
2014    4239480
2015    3895257
2016    3731944
2017    3533276
2018    3481663
2019    3436920
2020    1702158
2021    2916001
2022    3183260
2023    3418108
2024    3516677
Name: prestamos_presenciales, dtype: int64


## Tabla maestra → data/processed

Unidas las 5 fuentes a nivel distrito-año, con métricas derivadas (préstamos per
cápita, índice base 100 con base 2011). Guardamos la tabla maestra en `processed`.
Es la base para el análisis (PI1, PI2, PI3) de la Fase 3.

celda 44

In [35]:
maestra = maestra.sort_values(["Codi_Districte", "Any"]).reset_index(drop=True)

ruta = PROCESSED / "maestra_distrito_anyo.csv"
maestra.to_csv(ruta, index=False, encoding="utf-8")

print("Guardado:", ruta)
print("Filas:", len(maestra), "| Columnas:")
for c in maestra.columns:
    print("  -", c)

Guardado: C:\Users\User\Desktop\proyecto_bibliotecas_bcn\data\Proyecto Final\Fase 02\processed\maestra_distrito_anyo.csv
Filas: 150 | Columnas:
  - Codi_Districte
  - Nom_Districte
  - Any
  - prestamos_presenciales
  - poblacion
  - pct_mayores
  - pct_joven
  - pct_extranjera
  - renta_per_capita
  - prestamos_per_capita
  - indice_base100


## Verificación final de la tabla maestra

Comprobamos la integridad antes de cerrar la Fase 2: clave (distrito, año) única,
10 distritos × 15 años, y rangos plausibles de todas las métricas (% entre 0-100,
per cápita positivo, renta en miles, etc.).

celda 46

In [36]:
#celda 47

print("¿Clave (distrito, año) sin duplicados?:", maestra.duplicated(["Codi_Districte", "Any"]).sum() == 0)
print("Distritos:", maestra["Codi_Districte"].nunique(), "| Años:", maestra["Any"].nunique(), "| Filas:", len(maestra))

print("\nRangos de las métricas:")
cols = ["prestamos_per_capita", "indice_base100", "pct_mayores", "pct_joven", "pct_extranjera", "renta_per_capita"]
print(maestra[cols].describe().round(1).T[["min", "mean", "max"]])

¿Clave (distrito, año) sin duplicados?: True
Distritos: 10 | Años: 15 | Filas: 150

Rangos de las métricas:
                          min     mean      max
prestamos_per_capita      0.4      2.1      4.6
indice_base100           24.3     76.1    148.9
pct_mayores              11.5     21.0     26.7
pct_joven                26.5     28.7     33.5
pct_extranjera           11.0     19.9     54.0
renta_per_capita      13831.0  21776.4  36880.0
